**SAPIANS** · SCC5819 · MÓDULO 00 — A BASE SRAG · AUDITORIA DAS FAMÍLIAS

# Como conferir uma escolha por log loss sem treinar de novo

O walkthrough explica a comparação; este caderno verifica o protocolo, a
amostra, as configurações, os mínimos de validação e relações entre métricas.
Não treina modelos, não lê previsões individuais e não chama a leitura do
agregado de reprodução das previsões do preditor.

[README do módulo](../README.md) · [protocolo](../COURSE_STUDY.md) ·
[walkthrough](srag_sample_family_walkthrough.ipynb)

---

**Objetivos.** Reconstruir os identificadores e a grade; distinguir reuso
auditado de novo ajuste; conferir a escolha sem arredondar; separar validação,
teste exploratório e evidência necessária para reconstruir previsões.

**Estado e rota.** O registro distribuído desta auditoria pública da comparação entre famílias está concluído.
§§1–3 proveniência e configurações; §§4–6 contas e diagnósticos; §7 síntese. O setup informa o estado do arquivo efetivamente aberto; as guardas
para outras etapas continuam ativas. A unidade é a notificação, e o teste
histórico já observado permanece exploratório. Não há nova adoção de modelo.


In [1]:
%pip install -q numpy pandas pyarrow scikit-learn xgboost matplotlib imbalanced-learn joblib


Note: you may need to restart the kernel to use updated packages.


In [2]:
import ast
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import os
import pathlib
import sys
import tempfile

import matplotlib.pyplot as plt
from sklearn.metrics import log_loss
from sklearn.model_selection import ParameterGrid, ParameterSampler
import numpy as np
import pandas as pd

RANDOM_STATE = 42
REPO = pathlib.Path(
    os.environ.get("INTERPRETABLE_ML_REPO", pathlib.Path.cwd())
).resolve()
while REPO != REPO.parent and not (REPO / "tools/sapians.py").exists():
    REPO = REPO.parent
if not (REPO / "tools/sapians.py").exists():
    raise RuntimeError("Abra este caderno em um checkout do repositório.")
RECORD = pathlib.Path(
    os.environ.get(
        "SAPIANS_SAMPLE_FAMILY_RECORD",
        REPO / "modules/00-dataset/gold/sample_family_search.json",
    )
).resolve()
work_env = os.environ.get("INTERPRETABLE_ML_WORK")
WORK = (
    pathlib.Path(work_env).resolve()
    if work_env
    else pathlib.Path(tempfile.mkdtemp(prefix="srag_sample_family_internals-"))
)
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(REPO / "tools"))
import sapians as SP
# Sem figuras: sapians é usado para formatar as tabelas.

r = json.loads(RECORD.read_text()) if RECORD.exists() else None
protocol = r.get("protocol") if r else None
trials = r.get("validation_candidates", []) if r else []
selection = r.get("selection") if r else None
evaluation = r.get("evaluation") if r else None
status = r.get("status") if r else "absent"
if status not in {"absent", "running", "selected", "complete"}:
    raise ValueError("Estado desconhecido do registro da comparação.")
FINAL = status == "complete"
if FINAL and (not selection or not evaluation):
    raise ValueError("Registro completo sem seleção ou avaliação.")
ESTADOS = {
    "absent": "evidência ainda ausente",
    "running": "comparação em andamento",
    "selected": "seleção congelada; avaliação pendente",
    "complete": "comparação concluída",
}
ROTULOS = {
    "dummy": "Constante",
    "lpm": "Linear em probabilidade",
    "logit": "Regressão logística",
    "arvore": "Árvore de decisão",
    "floresta": "Random forest",
    "xgb": "XGBoost",
}
CURTOS = {
    "dummy": "Constante",
    "lpm": "Linear",
    "logit": "Logística",
    "arvore": "Árvore",
    "floresta": "Random forest",
    "xgb": "XGBoost",
}
ESTILOS = {
    "dummy": (SP.CINZA_CLARO, "--", "x"),
    "lpm": (SP.CINZA, ":", "s"),
    "logit": (SP.AMBAR, "-.", "^"),
    "arvore": (SP.ESCURO, ":", "D"),
    "floresta": (SP.SAGE, "--", "v"),
    "xgb": (SP.AZUL, "-", "o"),
}
print("Estado:", ESTADOS[status])
print("Fonte agregada:", RECORD.name)
print(
    "Este caderno lê evidência pública; não treina modelos nem carrega previsões individuais."
)
if protocol:
    print("Versão:", protocol["version"])
    print("SHA-256 do protocolo:", protocol["protocol_sha256"])
else:
    print("Nenhum resultado é preenchido sem o registro do estudo.")


Estado: comparação concluída
Fonte agregada: sample_family_search.json
Este caderno lê evidência pública; não treina modelos nem carrega previsões individuais.
Versão: course-sample-family-logloss-v1
SHA-256 do protocolo: 636a2f169424c2a3dcc320d5c4d0bd06b18f89cb280edccd2991c5d588a5073d


## §1 — Hashes identificam a receita e a amostra

O hash de um protocolo confere sua consistência, não sua verdade científica.
Aqui o recalculamos sem importar o executor da busca. Comparamos as fontes
científicas e as versões registradas com o checkout e o ambiente em uso. O lock
original é conferido separadamente, na cópia de proveniência preservada; o lock
ativo pode diferir apenas no pypdf, ferramenta de leitura do dicionário oficial.
Essa diferença é exibida, não tratada como igualdade entre ambientes.

Ler algumas colunas administrativas da amostra permite conferir calendário e
contagens sem exibir fichas nem probabilidades individuais. Auditar estes
registros não reexecuta os ajustes nem reproduz as previsões individuais. Para
isso, a [receita histórica](../provenance/README.md) exige o checkout e o lock
originais, além dos artefatos e dados correspondentes.


In [3]:
def sha_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for part in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(part)
    return digest.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()

def audit_signed(value, key):
    assert object_hash({k: v for k, v in value.items() if k != key}) == value[key]

def lock_entries(path):
    # Compare requirements and hashes; resolver comments are not dependencies.
    entries = {}
    current = None
    for raw in path.read_text().splitlines():
        line = raw.strip().removesuffix("\\").strip()
        if not line or line.startswith("#"):
            continue
        if raw[0].isspace():
            assert current is not None and line.startswith("--hash=sha256:"), "Linha inesperada no lock"
            entries[current].append(line)
        else:
            assert "==" in line, "Dependência sem versão exata no lock"
            current = line.split("==", 1)[0].lower().replace("_", "-")
            assert current not in entries, "Dependência duplicada no lock"
            entries[current] = [line]
    return entries

if protocol:
    audit_signed(protocol, "protocol_sha256")
    assert protocol["primary_metric"] == "logloss" and protocol["strategy"] == "plain"
    historical_lock = REPO / "modules/00-dataset/provenance/requirements-study.lock"
    for name, expected in protocol["code_sha256"].items():
        path = historical_lock if name == "requirements.lock" else REPO / "tools" / name
        assert sha_file(path) == expected, f"Fonte histórica difere do protocolo: {name}"
    for package, version in protocol["versions"].items():
        actual = platform.python_version() if package == "python" else importlib.metadata.version(package)
        assert actual == version, f"Ambiente científico difere do protocolo: {package}"
    historical_entries = lock_entries(historical_lock)
    active_lock = REPO / "requirements.lock"
    active_entries = lock_entries(active_lock)
    assert historical_entries.keys() == active_entries.keys(), "O lock ativo adicionou ou removeu dependências"
    changed = sorted(name for name in historical_entries if historical_entries[name] != active_entries[name])
    assert set(changed) <= {"pypdf"}, "O lock ativo mudou dependências além da ferramenta pypdf"
    print("SHA-256 do lock histórico:", sha_file(historical_lock))
    print("Lock ativo idêntico ao histórico:", sha_file(active_lock) == sha_file(historical_lock))
    print("Blocos de dependências diferentes no lock ativo:", ", ".join(changed) or "nenhum")
    print("pypdf no lock histórico:", historical_entries["pypdf"][0])
    print("pypdf no lock ativo:", active_entries["pypdf"][0])
    print("Versões científicas registradas: conferem com o ambiente em uso.")
    print("Auditoria do registro histórico; os ajustes e as previsões individuais não foram reexecutados.")
    assert sha_file(REPO / "modules/00-dataset/gold/counts.json") == protocol["cohort_manifest_sha256"]
    sample = REPO / "modules/00-dataset/gold/gold_covid_obito_sample.parquet"
    assert sha_file(sample) == protocol["source"]["sha256"]
    committed = subprocess.run(["git", "show", "HEAD:" + sample.relative_to(REPO).as_posix()],
                               cwd=REPO, check=True, capture_output=True).stdout
    assert hashlib.sha256(committed).hexdigest() == protocol["source"]["sha256"]
    assert len(committed) == sample.stat().st_size
    print("Tamanho em bytes da amostra comprometida:", SP.pt_int(len(committed)))
    del committed
    cohort = pd.read_parquet(sample, columns=["split", "y_obito", "gold_id", "ano_onset"])
    assert cohort["gold_id"].is_unique and cohort["gold_id"].is_monotonic_increasing
    assert cohort["y_obito"].isin([0, 1]).all()
    assert set(cohort["split"].astype(str).unique()) == {"train", "val", "test"}
    linhas = []
    for split, lower, upper in (("train", 2020, 2022), ("val", 2023, 2023), ("test", 2024, 2024)):
        part = cohort.loc[cohort["split"].eq(split)]
        saved = protocol["splits"][split]
        assert len(part) == saved["n"] and part["ano_onset"].between(lower, upper).all()
        assert object_hash(part["gold_id"].tolist()) == saved["ids_sha256"]
        if "events" in saved:
            assert int(part["y_obito"].sum()) == saved["events"]
            assert object_hash(part["y_obito"].tolist()) == saved["y_sha256"]
        linhas.append({"partição": split, "registros": len(part), "óbitos registrados": int(part["y_obito"].sum())})
    print(SP.tabela(pd.DataFrame(linhas)))
    print("Hashes da amostra, fontes científicas e lock histórico, versões registradas e composição das partições: conferem.")
    print("SHA-256 do arquivo da amostra:", protocol["source"]["sha256"])
    print("Entradas brutas:", SP.pt_int(len(protocol["features"])))
    assert len(protocol["features"]) == 40
    print("Semente:", SP.pt_int(protocol["seed"]))
else:
    print("PENDENTE — hashes e contagens não são presumidos.")


SHA-256 do lock histórico: f477ce0c78a92d0adc908a008fe0313c387765ac6024a5dc9f814d2a8d8f01fe
Lock ativo idêntico ao histórico: False
Blocos de dependências diferentes no lock ativo: pypdf
pypdf no lock histórico: pypdf==6.15.0
pypdf no lock ativo: pypdf==6.21.1
Versões científicas registradas: conferem com o ambiente em uso.
Auditoria do registro histórico; os ajustes e as previsões individuais não foram reexecutados.
Tamanho em bytes da amostra comprometida: 4.006.315
partição registros óbitos registrados
   train   200.000             62.719
     val    24.148              4.572
    test    16.142              2.945
Hashes da amostra, fontes científicas e lock histórico, versões registradas e composição das partições: conferem.
SHA-256 do arquivo da amostra: cb79c99338a5a7b3c125659aad148a73749e2695f2048c9481a20ab29e3ca918
Entradas brutas: 40
Semente: 42


## §2 — Reconstruir a grade sem chamar a busca

Usamos os espaços e orçamentos registrados para reconstruir a grade ou
amostragem de configurações. A semente determina a amostra da grade; o JSON
canônico determina o identificador. Esta reconstrução não executa ajustes.
O orçamento diferente por família deve ser visível, e todos os candidatos
devem permanecer sem balanceamento.


In [4]:
if protocol:
    specs = protocol["candidates"]
    assert len({row["candidate_id"] for row in specs}) == len(specs)
    assert set(protocol["budgets"]) == set(ROTULOS)
    assert all(row["strategy"] == "plain" for row in specs)
    assert len(set(protocol["features"])) == len(protocol["features"])
    source_tree = ast.parse((REPO / "tools/srag_50_selection.py").read_text())
    source_values = {}
    for node in source_tree.body:
        targets = node.targets if isinstance(node, ast.Assign) else [node.target] if isinstance(node, ast.AnnAssign) else []
        for target in targets:
            if isinstance(target, ast.Name) and target.id in {"ESPACOS", "N_ITER"}:
                source_values[target.id] = ast.literal_eval(node.value)
    assert source_values["ESPACOS"] == protocol["parameter_spaces"]
    assert source_values["N_ITER"] == protocol["budgets"]
    reconstruidos = []
    for family, budget in protocol["budgets"].items():
        space = protocol["parameter_spaces"][family]
        grid = list(ParameterGrid(space)) if space else [{}]
        if len(grid) > budget:
            grid = list(ParameterSampler(space, n_iter=budget, random_state=protocol["seed"]))
        for params in grid:
            spec = {"family": family, "strategy": "plain", "params": params}
            spec["candidate_id"] = object_hash(spec)[:16]
            reconstruidos.append(spec)
    assert {row["candidate_id"]: row for row in reconstruidos} == {row["candidate_id"]: row for row in specs}
    by_spec = {row["candidate_id"]: row for row in specs}
    assert len({row["candidate_id"] for row in trials}) == len(trials)
    if FINAL:
        assert {row["candidate_id"] for row in trials} == set(by_spec)
    for row in trials:
        assert row["candidate_id"] in by_spec
        assert all(row[k] == by_spec[row["candidate_id"]][k] for k in ("family", "strategy", "params"))
        assert row["status"] == "complete" and row["protocol_sha256"] == protocol["protocol_sha256"]
        audit_signed(row, "result_sha256")
    linhas = [{"família": ROTULOS[family], "registradas": budget,
               "concluídas": sum(row["family"] == family for row in trials)} for family, budget in protocol["budgets"].items()]
    print(SP.tabela(pd.DataFrame(linhas)))
    print("Configurações reconstruídas:", SP.pt_int(len(reconstruidos)))
    print("Identificadores e resultados assinados correspondem ao protocolo:", True)
else:
    print("PENDENTE — nenhuma grade é inventada antes de existir o protocolo.")


                família registradas concluídas
      Árvore de decisão          24         24
              Constante           1          1
          Random forest          10         10
    Regressão logística           7          7
Linear em probabilidade           1          1
                XGBoost          30         30
Configurações reconstruídas: 73
Identificadores e resultados assinados correspondem ao protocolo: True


## §3 — Reutilização precisa preservar a proveniência

O reuso do braço sem balanceamento do XGBoost conserva os resultados da busca
anterior, mas os vincula ao novo protocolo. Comparamos especificações, métricas
inteiras e assinaturas de origem ao agregado público anterior. O exportador
também confere arquivos privados de previsões e o pipeline salvo antes da
promoção; este reader não repete essa execução nem lê esses arquivos individuais.


In [5]:
if protocol:
    reutilizados = [row for row in trials if row["origin"]["kind"] == "imported_phase53_plain"]
    novos = [row for row in trials if row["origin"]["kind"] == "fitted_phase54"]
    assert len(reutilizados) + len(novos) == len(trials)
    if protocol["import_xgb"]:
        anterior = json.loads((REPO / "modules/00-dataset/gold/sample_balance_search.json").read_text())
        audit_signed(anterior["protocol"], "protocol_sha256")
        assert anterior["status"] == "complete"
        assert anterior["protocol"]["source"] == protocol["source"]
        assert anterior["protocol"]["features"] == protocol["features"]
        assert anterior["protocol"]["versions"] == protocol["versions"]
        assert anterior["protocol"]["seed"] == protocol["seed"]
        antigos = {row["candidate_id"]: row for row in anterior["validation_candidates"] if row["strategy"] == "plain"}
        atuais = {row["candidate_id"]: row for row in protocol["candidates"] if row["family"] == "xgb"}
        assert set(antigos) == set(atuais)
        assert len(reutilizados) == protocol["import_xgb"]["candidate_count"]
        for row in reutilizados:
            old = antigos[row["candidate_id"]]
            audit_signed(old, "result_sha256")
            assert row["family"] == "xgb" and row["strategy"] == "plain"
            assert all(row[k] == old[k] for k in ("params", "validation", "effective_params", "fit_warnings"))
            assert row["origin"]["protocol_sha256"] == anterior["protocol"]["protocol_sha256"]
            assert row["origin"]["result_sha256"] == old["result_sha256"]
    else:
        assert not reutilizados
    print("Resultados reutilizados e auditados:", SP.pt_int(len(reutilizados)))
    print("Novos ajustes registrados:", SP.pt_int(len(novos)))
    print("Reuso não significa refazer o treino; não foram copiados resultados do teste.")
else:
    print("PENDENTE — nenhum reuso é afirmado sem registro.")


Resultados reutilizados e auditados: 30
Novos ajustes registrados: 43
Reuso não significa refazer o treino; não foram copiados resultados do teste.


## §4 — Conferir a aritmética sem confundir perda com probabilidade

O ensaio abaixo usa valores artificiais para comparar a fórmula do log loss
com a biblioteca. Depois conferimos relações internas das métricas agregadas
reais: contagens, frequência, médias e calibração por faixas. Isso detecta
inconsistências, mas não calcula novamente o log loss real sem suas previsões.


In [6]:
y_demo = np.array([0, 1, 0, 1])
p_demo = np.array([0.1, 0.8, 0.2, 0.7])
manual = float(-np.log(np.where(y_demo == 1, p_demo, 1 - p_demo)).mean())
library = float(log_loss(y_demo, p_demo, labels=[0, 1]))
np.testing.assert_allclose(manual, library, rtol=0, atol=1e-14)
print("Ensaio artificial · log loss manual:", SP.pt(manual, 8))
print("Ensaio artificial · log loss da biblioteca:", SP.pt(library, 8))
print("Esse ensaio não é uma medição dos dados SRAG.")

def audit_metrics(m):
    n, events = m["n"], m["events"]
    assert n > 0 and 0 < events < n and np.isfinite(m["logloss"]) and m["logloss"] >= 0
    assert all(0 <= m[key] <= 1 for key in ("brier", "auc", "ap", "mean_predicted", "observed", "ece_10_equal_width"))
    np.testing.assert_allclose(m["observed"], events / n, rtol=0, atol=1e-12)
    np.testing.assert_allclose(m["calibration_gap"], m["mean_predicted"] - m["observed"], rtol=0, atol=1e-12)
    bins = m["reliability"]
    assert len({b["bin"] for b in bins}) == len(bins)
    assert set(b["bin"] for b in bins).issubset(range(10))
    assert sum(b["n"] for b in bins) == n
    assert all(b["n"] > 0 and 0 <= b["mean_predicted"] <= 1 and 0 <= b["observed"] <= 1 for b in bins)
    np.testing.assert_allclose(sum(b["n"] * b["mean_predicted"] for b in bins) / n, m["mean_predicted"], rtol=0, atol=1e-12)
    np.testing.assert_allclose(sum(b["n"] * b["observed"] for b in bins) / n, m["observed"], rtol=0, atol=1e-12)
    ece = sum(b["n"] * abs(b["mean_predicted"] - b["observed"]) for b in bins) / n
    np.testing.assert_allclose(ece, m["ece_10_equal_width"], rtol=0, atol=1e-12)
    assert m["tp"] + m["fn"] == events and m["tn"] + m["fp"] == n - events
    np.testing.assert_allclose(m["sensitivity"], m["tp"] / events, rtol=0, atol=1e-12)
    np.testing.assert_allclose(m["specificity"], m["tn"] / (n - events), rtol=0, atol=1e-12)
    if m["tp"] + m["fp"]:
        np.testing.assert_allclose(m["precision"], m["tp"] / (m["tp"] + m["fp"]), rtol=0, atol=1e-12)
    else:
        assert m["precision"] is None

for row in trials:
    audit_metrics(row["validation"])
    assert row["validation"]["n"] == protocol["splits"]["val"]["n"]
    assert row["training"]["n"] == protocol["splits"]["train"]["n"]
    assert row["training"]["events"] == protocol["splits"]["train"]["events"]
    if row["family"] == "logit":
        diagnostic = row["convergence"]
        assert diagnostic["converged"] is True
        assert diagnostic["n_iter"] and all(0 < n < diagnostic["max_iter"] for n in diagnostic["n_iter"])
print("Resultados de validação com aritmética conferida:", SP.pt_int(len(trials)))


Ensaio artificial · log loss manual: 0,22708064
Ensaio artificial · log loss da biblioteca: 0,22708064
Esse ensaio não é uma medição dos dados SRAG.
Resultados de validação com aritmética conferida: 73


## §5 — O mínimo precisa existir antes da avaliação do teste

Reconstruímos a escolha pelo valor registrado sem arredondar. Cada finalista
é o mínimo de sua família; o vencedor é o mínimo entre todos os candidatos.
Os identificadores resolvem somente empates numéricos exatos. Esse procedimento
não é um teste estatístico de superioridade entre modelos próximos. O menor
valor da validação resulta de uma escolha entre candidatos e não deve ser
tratado como avaliação independente dessa escolha.


In [7]:
if selection:
    audit_signed(selection, "selection_sha256")
    assert selection["protocol_sha256"] == protocol["protocol_sha256"]
    assert len(trials) == len(protocol["candidates"]) == selection["candidate_count"]
    assert selection["test_used_for_selection"] is False
    assert selection["finalists_frozen_before_test"] is True
    assert selection["probabilities_recalibrated"] is False
    assert set(row["family"] for row in selection["finalists"]) == set(protocol["budgets"])
    for final in selection["finalists"]:
        rows = [row for row in trials if row["family"] == final["family"]]
        expected = min(rows, key=lambda row: (row["validation"]["logloss"], row["candidate_id"]))
        assert final == expected
    expected = min(trials, key=lambda row: (row["validation"]["logloss"], row["candidate_id"]))
    assert selection["winner"] == expected
    print("Finalistas correspondem aos mínimos sem arredondar:", True)
    print("Família selecionada:", ROTULOS[selection["winner"]["family"]])
    print("Log loss de validação do selecionado:", SP.pt(selection["winner"]["validation"]["logloss"], 8))
    print("O teste foi usado na regra de seleção:", False)
else:
    print("PENDENTE — seleção requer todos os candidatos concluídos.")

if evaluation:
    audit_signed(evaluation, "evaluation_sha256")
    assert selection and evaluation["protocol_sha256"] == protocol["protocol_sha256"]
    assert evaluation["selection_sha256"] == selection["selection_sha256"]
    assert evaluation["selection_changed"] is False and evaluation["finalists_frozen_before_test"] is True
    assert evaluation["canonical_model"] == selection["canonical_model"]
    assert [row["candidate_id"] for row in evaluation["finalists"]] == [row["candidate_id"] for row in selection["finalists"]]
    for row in evaluation["finalists"]:
        audit_metrics(row["test"])
        assert row["test"]["n"] == protocol["splits"]["test"]["n"]
    winner_test = next(row["test"] for row in evaluation["finalists"] if row["candidate_id"] == selection["winner"]["candidate_id"])
    assert evaluation["test"] == winner_test
    print("Avaliação preserva os finalistas e o vencedor congelados:", True)
    print("Log loss do vencedor no teste exploratório:", SP.pt(winner_test["logloss"], 8))
else:
    print("PENDENTE — nenhuma conclusão de teste antes de existir a seleção.")


Finalistas correspondem aos mínimos sem arredondar: True
Família selecionada: XGBoost
Log loss de validação do selecionado: 0,42090336
O teste foi usado na regra de seleção: False
Avaliação preserva os finalistas e o vencedor congelados: True
Log loss do vencedor no teste exploratório: 0,40901958


## §6 — Diagnósticos de ajuste e o limite da auditoria pública

Uma logística que não convergiu não deve receber rótulo de candidato completo.
O modelo linear de probabilidade é avaliado com limite explícito nas saídas;
esse tratamento precisa ser visível. Registramos os diagnósticos disponíveis
sem tratar a ausência de um aviso como prova de validade clínica.


In [8]:
if FINAL:
    def audit_clip(diagnostic, expected_n):
        assert diagnostic["n"] == expected_n
        assert diagnostic["clip_lower"] == 1e-6 and diagnostic["clip_upper"] == 1 - 1e-6
        assert np.isfinite(diagnostic["raw_min"]) and np.isfinite(diagnostic["raw_max"])
        assert diagnostic["raw_min"] <= diagnostic["raw_max"]
        assert 0 <= diagnostic["raw_below_zero"] <= diagnostic["clipped_low"] <= expected_n
        assert 0 <= diagnostic["raw_above_one"] <= diagnostic["clipped_high"] <= expected_n
        assert diagnostic["clipped_low"] + diagnostic["clipped_high"] <= expected_n
        assert (diagnostic["raw_min"] < 0) == (diagnostic["raw_below_zero"] > 0)
        assert (diagnostic["raw_max"] > 1) == (diagnostic["raw_above_one"] > 0)

    for row in selection["finalists"]:
        if row["family"] == "lpm":
            assert row.get("lpm_clip") is not None
            audit_clip(row["lpm_clip"], protocol["splits"]["val"]["n"])
            print("Limites de avaliação do baseline linear: [0,000001; 0,999999].")
            print("Saídas lineares · validação:")
            print(SP.tabela(pd.DataFrame([row["lpm_clip"]]), {"clip_lower": 6, "clip_upper": 6}))
            test_row = next(x for x in evaluation["finalists"] if x["family"] == "lpm")
            assert test_row.get("lpm_clip") is not None
            audit_clip(test_row["lpm_clip"], protocol["splits"]["test"]["n"])
            print("Saídas lineares · teste exploratório:")
            print(SP.tabela(pd.DataFrame([test_row["lpm_clip"]]), {"clip_lower": 6, "clip_upper": 6}))
        if row["family"] == "logit":
            assert row.get("convergence") is not None
            print("Diagnóstico registrado de convergência da logística:")
            print(SP.tabela(pd.DataFrame([row["convergence"]]), 4))
    print("O teste já havia sido observado; não é uma avaliação confirmatória nova.")
    print("A comparação de balanceamento do XGBoost permanece separada desta seleção entre famílias.")
else:
    print("PENDENTE — diagnósticos finais ainda não disponíveis.")
print("Não foram ajustados modelos nem lidos vetores de previsões individuais neste caderno.")
print("Integridade e coerência do agregado não equivalem a reconstruir as previsões salvas.")


Limites de avaliação do baseline linear: [0,000001; 0,999999].
Saídas lineares · validação:
clip_lower clip_upper clipped_high clipped_low      n raw_above_one raw_below_zero raw_max raw_min
  0,000001   0,999999            6       4.993 24.148             6          4.993  1,2987 -0,4592
Saídas lineares · teste exploratório:
clip_lower clip_upper clipped_high clipped_low      n raw_above_one raw_below_zero raw_max raw_min
  0,000001   0,999999            1       3.817 16.142             1          3.817  1,0416 -0,4871
Diagnóstico registrado de convergência da logística:
 converged max_iter n_iter
      True    2.000  [147]
O teste já havia sido observado; não é uma avaliação confirmatória nova.
A comparação de balanceamento do XGBoost permanece separada desta seleção entre famílias.
Não foram ajustados modelos nem lidos vetores de previsões individuais neste caderno.
Integridade e coerência do agregado não equivalem a reconstruir as previsões salvas.


## §7 — O que o livro diz e como continuar

Na PFI de Molnar, o modelo já foi treinado e continua fixo durante a permutação.
Escolher um preditor por log loss acontece antes de interpretar sua dependência
das variáveis. A auditoria deste protocolo não altera automaticamente o modelo
da apresentação e não substitui uma avaliação clínica.

Adoção de outro pipeline exige decisão explícita, novas previsões e recomputação
das explicações afetadas. Guardamos o histórico por ROC-AUC como histórico,
sem chamar números antigos arredondados de nova seleção por log loss.

Fontes: [protocolo do módulo](../COURSE_STUDY.md),
[Molnar, PFI](https://christophm.github.io/interpretable-ml-book/feature-importance.html),
[scikit-learn, avaliação](https://scikit-learn.org/stable/modules/model_evaluation.html) e
[scikit-learn, seleção e avaliação](https://scikit-learn.org/stable/modules/cross_validation.html).

**Encerramento.** As verificações e comparações pertencem ao protocolo e ao
agregado identificados no setup. A referência histórica das aulas permanece
separada; adotar outro pipeline exigiria uma decisão própria e novas explicações.
